# Sentiment Analysis using Hugging Face Transformers

This notebook explores how to use a pre-trained transformer model to perform sentiment analysis on text data.  
We start with basic inference and progressively adapt the pipeline toward aspect-based sentiment analysis.

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [108]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import numpy as np
from scipy.special import softmax
import csv
import urllib.request
import pandas as pd
import time

## Model Overview

We use the model:

**cardiffnlp/twitter-roberta-base-sentiment**

### Key characteristics:
- Based on the RoBERTa architecture
- Fine-tuned on Twitter data
- Outputs 3 sentiment classes:
  - Negative
  - Neutral
  - Positive


## 1. Model Initialization & Basic Handling

In [4]:
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = "@user" if t.startswith("@") and len(t) > 1 else t
        t = "http" if t.startswith("http") else t
        new_text.append(t)
    return " ".join(new_text)

In [5]:
task = "sentiment"
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

labels = []
mapping_link = f"https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/{task}/mapping.txt"

Loading weights: 100%|█████████████████| 201/201 [00:00<00:00, 17442.64it/s]


In [6]:
with urllib.request.urlopen(mapping_link) as f:
    html = f.read().decode("utf-8").split("\n")
    csvreader = csv.reader(html, delimiter="\t")
    labels = [row[1] for row in csvreader if len(row) > 1]

In [15]:
text = "This is very bad food"

In [34]:
text = preprocess(text)
encoded_input = tokenizer(text, return_tensors="pt")
output = model(**encoded_input)

scores = output.logits[0].detach().numpy()
scores = softmax(scores)
ranking = np.argsort(scores)[::-1]

for i in range(scores.shape[0]):
    label = labels[ranking[i]]
    score = scores[ranking[i]]
    print(f"{i+1}) {label}: {np.round(float(score), 4)}")

1) negative: 0.975
2) neutral: 0.0215
3) positive: 0.0034


In [45]:
def predict(text):
    text = preprocess(text)
    encoded_input = tokenizer(text, return_tensors="pt")
    output = model(**encoded_input)
    scores = output.logits[0].detach().numpy()
    scores = softmax(scores)
    ranking = np.argsort(scores)[::-1]
    # Extract the top score and label
    label = labels[ranking[0]]
    score = scores[ranking[0]]
    return label, float(f"{score:.2f}")

In [46]:
print(predict("This is very bad food"))

('negative', 0.98)


## 2. Testing on Multiple Comments

We test the model on a list of different comments to evaluate its behavior  
on real-world-like inputs.

In [25]:
Data = [
    {"comment" : "Your attention to detail on this project was truly impressive." , "label" : "positive" },
    {"comment" :"The weather today is absolutely perfect for a walk in the park.", "label" : "positive" },
    {"comment" :"You have a natural talent for making everyone in the room feel welcome." ,"label" :  "positive"},
    {"comment" :"The service at the restaurant was incredibly slow and frustrating.","label" : "negative"},
    {"comment" :"This smartphone battery barely lasts half a day, which is very disappointing.","label" : "negative"},
    {"comment" :"The instructions for the new software are confusing and difficult to follow." ,"label" :  "negative"}    
]

In [49]:
results = []
for d in Data:
    predicted_label , score = predict(d["comment"])
    r= {"comment" : d["comment"] ,
        "label" : d["label"] ,
        "predicted_label" : predicted_label ,
        "score" : score
       }
    results.append(r)

df = pd.DataFrame(results)
df

,comment,label,predicted_label,score
0,Your attention to detail on this project was t...,positive,positive,0.98
1,The weather today is absolutely perfect for a ...,positive,positive,0.99
2,You have a natural talent for making everyone ...,positive,positive,0.97
3,The service at the restaurant was incredibly s...,negative,negative,0.98
4,This smartphone battery barely lasts half a da...,negative,negative,0.98
5,The instructions for the new software are conf...,negative,negative,0.89


## 3. Handling Complex and Ambiguous Sentences

Some comments contain mixed opinions (e.g., positive + negative in the same sentence).  
This creates ambiguity for standard sentiment models.

Example:
"The design is great but the price is too high."

In such cases, the model returns a single sentiment,  
which may not reflect the true meaning of the comment.

In [50]:
Tricky_sentences =[
    "I really liked the design and the idea behind it, but the execution was honestly disappointing.",
    "The service started off great and the staff were friendly, but everything went downhill by the end.",
    "At first I thought this was a complete waste of time, but it actually turned out to be quite useful.",
    "The beginning was slow and confusing, but in the end it became really interesting and enjoyable."
]

In [51]:
for s in Tricky_sentences:
    print(predict(s))

('negative', 0.63)
('negative', 0.58)
('positive', 0.64)
('positive', 0.93)


## 4. Introduction of Aspect-Based Sentiment Analysis

To address the limitations of global sentiment classification,  
we introduce aspect-based sentiment analysis.

Instead of analyzing the whole sentence,  
we focus on specific aspects such as:
- Price
- Design
- Quality
- Service

This allows more granular and meaningful insights.

In [52]:
# Define possible aspects 
ASPECTS = ["design", "performance", "price", "service", "quality", "delivery", "staff"]

In [76]:
def check_aspect(sentence):
    founded = False
    asp = "Other"
    for aspect in ASPECTS:
        if aspect.lower() in sentence.lower():
            asp = aspect.lower()
            founded = True
    return founded , asp

In [102]:
def split_comment(text):
    text_lower = text.lower()
    if " but " in text_lower or " however " in text_lower:
        parts = text.replace(" but ", ". ").replace(" however ", ". ").split(".")
    else:
        parts = [text]
    # remove empty parts
    parts = [p.strip() for p in parts if p.strip()]
    return parts

In [104]:
# After fixing the empty sentences issue
def aspect_based_sentiment(text):
    results_with_aspects = []
    results_without_aspects = []
    parts = split_comment(text)
    for p in parts:
        label, score = predict(p)
        founded, asp = check_aspect(p)
        if founded:
            results_with_aspects.append({"Sentence": p,"Aspect": asp,"Label": label,"Score": score})
        else:
            results_without_aspects.append({"Sentence": p,"Label": label,"Score": score})
    return results_with_aspects, results_without_aspects

In [90]:
results_with_aspects ,results_without_aspects =  aspect_based_sentiment("I really liked the design and the idea behind it, but the execution was honestly disappointing")
print(results_with_aspects)
print(results_without_aspects)

[{'Sentence': 'I really liked the design and the idea behind it,', 'Aspect': 'design', 'Label': 'positive', 'Score': 0.98}]
[{'Sentence': ' the execution was honestly disappointing', 'Label': 'negative', 'Score': 0.98}]


## Observation

From this result, we notice that the model still struggles with comments  
that contain multiple opinions.

Even though aspects are detected, the sentiment is assigned at the phrase level  
without properly isolating each opinion when the sentence is complex.

This highlights a limitation in the current approach and motivates  
the need for a better handling of such cases.

In [106]:
results_with_aspects ,results_without_aspects =  aspect_based_sentiment("The design is beautiful.")
print(results_with_aspects)
print(results_without_aspects)

[{'Sentence': 'The design is beautiful.', 'Aspect': 'design', 'Label': 'positive', 'Score': 0.98}]
[]


## 5. Testing Across Different Scenarios

We test the pipeline on various types of inputs.

In [92]:
test_comments = [
    # Straightforward + known aspect
    "The design is beautiful.",
    "The price is too expensive.",

    # Straightforward + no known aspect
    "I really loved it.",
    "This was a complete waste of time.",

    # Positive then negative, both with known aspects
    "The design is beautiful but the performance is terrible.",
    "The staff were friendly but the service was very slow.",

    # Negative then positive, both with known aspects
    "The price is high but the quality is excellent.",
    "The performance was bad at first but the service was great.",

    # Complex: one part has aspect, one part has no aspect
    "The design is amazing but I still regret buying it.",
    "I hated the experience at first but the staff were very kind.",

    # Multiple aspects in one sentence part
    "The design and performance are both excellent.",
    "The price and delivery were disappointing.",

    # No aspect at all, mixed sentiment
    "I liked it at first but it became disappointing later.",
    "At the beginning it was confusing but in the end it was useful.",

    # Edge cases
    "The product is okay.",
    "Not bad, but not amazing either.",
    "The service was not terrible, but it was not great.",
    "The delivery was fast however the package looked damaged.",
]

In [112]:
r_aspects = []
r_w_aspects = []

start_time = time.perf_counter()
for comment in test_comments:
    r_a, r_w_a = aspect_based_sentiment(comment)
    if r_a:
        r_aspects.extend(r_a)
    if r_w_a: 
        r_w_aspects.extend(r_w_a)
end_time = time.perf_counter() 
print(f" This function took {end_time - start_time} s to be executed")

 This function took 74.58685916592367 s to be executed


In [117]:
df_aspects = pd.DataFrame(r_aspects)
df_aspects.head(5)

,Sentence,Aspect,Label,Score
0,The design is beautiful.,design,positive,0.98
1,The price is too expensive.,price,negative,0.86
2,The design is beautiful,design,positive,0.98
3,the performance is terrible,performance,negative,0.97
4,The staff were friendly,staff,positive,0.90


In [118]:
df_w_aspects = pd.DataFrame(r_w_aspects)
df_w_aspects.head(5)

,Sentence,Label,Score
0,I really loved it.,positive,0.98
1,This was a complete waste of time.,negative,0.98
2,I still regret buying it,negative,0.89
3,I hated the experience at first,negative,0.97
4,I liked it at first,positive,0.91


### Performance Observation & Time Analysis

During testing, we measure the execution time of the pipeline.

Observation:
- The model processes inputs sequentially
- Multiple predictions are generated for a single comment
- Execution time increases when handling multiple comments

Conclusion:
The pipeline shows a noticeable increase in processing time  
when dealing with a larger number of inputs.

This indicates that the current implementation is not optimized  
for scalability and requires improvement in handling multiple inputs efficiently.